# 🛡️ GhostSignal — SmolLM3-3B Offline Scam Shield Pipeline
### End-to-End Google Colab Fine-Tuning, Holdout Evaluation & GGUF Q4_K_M Quantization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/notebook#fileId=https%3A//huggingface.co/HuggingFaceTB/SmolLM3-3B.ipynb)

This notebook provides the **complete, zero-configuration pipeline** to fine-tune **HuggingFaceTB/SmolLM3-3B** on GhostSignal's 508-sample balanced cybersecurity scam dataset.

### 🎯 Pipeline Objectives
1. **Disguised Threat Detection**: Fake KYC expirations, OTP harvesting, UPI reverse collect traps, and electricity bill cutoffs.
2. **Unicode & Evasion Shield**: Dotted keywords (`b.a.n.k`, `k.y.c`), Cyrillic homoglyphs (`Yоur Cedаr Bank`), zero-width spaces, and split words.
3. **Structured JSON Verdicts**: Emits strict JSON schemas with `risk_level`, `risk_score`, `scam_type`, `red_flags`, `explanation`, and `recommended_action` (including India Cyber Fraud Helpline **1930** and `cybercrime.gov.in`).
4. **Quantization to GGUF (Q4_K_M, ~1.82 GB)**: Direct 3.3x compression for 100% offline, private CPU execution in GhostSignal.

> ⚡ **Hardware Requirement**: Google Colab Free Tier **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).

## Step 1: Hardware Environment & GPU Verification
Verify CUDA availability, GPU device name, and available VRAM.

In [ ]:
import torch
import sys

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ Active GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    raise RuntimeError("❌ No GPU detected. Please go to Runtime -> Change runtime type and select T4 GPU.")

## Step 2: Install Unsloth & Hugging Face Training Stack
Unsloth accelerates LoRA training by 5x and provides direct GGUF quantization.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub
print("Dependencies installed successfully!")

## Step 3: Load SmolLM3-3B in 4-bit with LoRA Adapters
We load `HuggingFaceTB/SmolLM3-3B` with 4-bit QLoRA and attach adapters to all projection modules (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`).

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None # Auto-detects float16 or bfloat16
load_in_4bit = True

print("Loading HuggingFaceTB/SmolLM3-3B in 4-bit...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "HuggingFaceTB/SmolLM3-3B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Configure LoRA parameter-efficient adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✅ SmolLM3-3B initialized with LoRA adapters (Rank 16, Alpha 32).")

## Step 4: Ingest 508-Sample Balanced Synthetic Scam Dataset
The complete dataset (408 train, 50 validation, 50 holdout test samples) is embedded directly into this notebook.
**No file upload is required.** It unpacks automatically into ChatML instruction-formatted records.

In [ ]:
import gzip
import base64
import json
from datasets import Dataset

TRAIN_B64 = "H4sIAH405WoC/+2d3XLbaHauz3dV7gHbJz2psjgACJCA5shb7nR3Mu32uN2TmkqnXBD5SUQEARz8SK2Zmqp9npvxFfgadA25kqwPICVSJinaEgmAeCqpZEaiSPD7ebx+3rXW319cqiwLzlX24tj4j7+/SJNIyX96kd1kubp88dJ4MUriXMW5/uFfksIIUmV8N0my/OfwPA4i49UPL40gNpKzsyiMlTG6OVVppkZFGuY3RjYKLo2xytUoD5PYuEzGKuoZr+Tvbv6mjHyijGmaXIVjNTZmj/Fy9nJjHGbnRZjJb/JJqoI8e2n8TaXJ0XU4zie/nySXyXl0M50YyelZIR+j319eMkrVWB42lAeTdz/L9bON5Znz8Cwc6Z8W6bmKRzc946cinxa5cZMUqXGl0nEonxnGRpan+j/9688/vTHOkvQyyI3rMJ8YF+omOzbSMLv4EKkrFRm/++NP//7S+PHb1z/88uNL4/sfvvv+n19Wv89GiazR78wjyzTlZ3oNPuQ3U/lq8nAfzqLgPDN+F4VZLotWfl58nsnrFr7I7OXqt2kUxOVPqu+RqlFyealiWa8PQbmkvRf/eGnc71qRqfTBnt0vdpjNF1l/s2pvwli+eJAnaXb8a/zrC/0/v7z77ts374+Nv+iVeZOk+STLg9Q47QW9uHdhBKNRUsR6VaLIOFXGaZSMLvQuJeNAlvXPKg3PboyL3k1vZMjiTfJ8mh3//vfTIB2p6OhMqZ76LbicRqqXqyyXn98YoXyncRjkKrrpVc+w/K2CLJPVCuT7LH+1v/8aG8avL+435Vf56a8v9F78+uLlwi/LHdG/HPqzH99tSvUnp0F88SG8nMrRTaoFv3+D+Z7pV/6H/pmx6Q/0L+8P4YdU/bWQr/khST/MFu5DdZwXXj47kvK95Qf/OfvYh6ehes5xkudqfPdsCwek+v1JFISXmeyFHJVpqjJ5CiMw9MOW50f/KCvk/5RXT05TOA31S+T1V9W+ybmQX+f6AMulnu+1PHJSnE/kreYHSO76xd1mz3+t31R/1jeZxkF134LpVL+p3jU5scZoEsSxZsDrJM+MayWfE8ZyaOVbyWHKr5WKjUjJl0zLbzGHgHyyXMDrJB33Fjbm4WWo1uB1YsSJfFIUji5eGnK+XuoHyCYaXPc7k/WMk9kz6cdO0vMgDv9WLuXd9wnzz7+IrIrcGTXVnxvn0U21cKE8/t1X++FMU8XIimyqQXYWxkFcvkXJRuMsDYqxkCqQ+2P5fXN2s6dy0/SFKV80SsNL1TtPrnphrK9E/I8X//jPf/zT//k7tIbWS7T+d1Xu8VmaXK6AdVZM9bnqVTCfX+dYqXEmj3dUnd3qW+grWV4c2fTg5vgO3LM/OpqfkmV8R8l5GO+Y2R7MhtkwG2YfCrNfK0H0qJAre6nSl8aJGst/P0kSIZM8ypW6x/d8l3N1t8uyXUVcZIU+3vKAVxrJ+lqchelltdZzAsjrgzDK5GZd3+M8nv8bUcF/GecY34AckANyQL4dyP9y+0mWRPh9+3EZ4P+vvPSzCyz/X97LkPcsT6V+pJ7xTk3l/JbrVa7r/J8D44fX5fOfJVGUXG8VPimBM8NG+VE7DqL4LTfIoyS5CKLwQh0Cyv8oX+ao/DZ36C59wUAwPFH66KbhNK9gX5SIWAC7ZnoG1IE6UF9rnd8HVUqo78AkL6bhUarOiniMMb41xOMkPgR+w17YC3tXRrPfykcoOWGvw/Mwl9Wr8Pv0gPYGU3q3yPVckAtyQS7IbRJy/yWMbz9GxnWQxvJkcrhuP61m77FRTMdBrox/+8tJuUiyfLYxkQXNyvuh0qtwJPf7MrjRuhDZ0OA00qGOWu1czyJYQbACekNv5B9fYy0XvWkPbJMrJFcIqSH1c8mqP5d8LGUMN2urtf29oKxeY1bvUVXddlgT2AC4ALdzKrtd5fNG+pOO9KU9Gk3U6GKfZjNZPUgMiSFxk0m8OrF3WBwm1QeH4TAc7jCH/+f//3cjUNxytXIYX4VZeBodRALwh/mX0T9K5Vc6fLwcXA5jeS79nrn6TS759SQcTQSAsTERmJRJwCpNnVwaWajPkEAz0u8D7aE9tF+kva5K0bi//dQZu9tB7YHaA6SD9O5o9VYHtlHrwW/4Db/hd5NN8k4kJX0XqAN1oA7UD9UoDxZt8mRH5TO1N+LzByQ1SWqCX/DbWF31DLw69bgU5W6psNofkrskdwnUgTpQv7emnziJYGMkZJ9wH1DiSIkj1IbaB9MOdXlizEE3QfVMgiEEQyAwBG6N3fxZJESHSbYNhjTFavYIiRASAe2gvePG9cMgyMHZ10RHiI4AcABOdGRLeq9LV+6d3DaRESIj0Bf6Nr9RqoawjoI8R6PUWuplXKIhREPAOTjv5OTFGch1Uc2zm9MbB+Xuf+aiTQ0NNTTgHJyTt4xal7T0HUIihESALtDtxESCLSoV96kYAb7AF/gCX3piN6GInOwgNIbG0Lh52ozVJrBODx5uXLntYg0SiCQQIT7Ery/4of99aGqtDHAH7sAduCP2u1Nct1bpx4xd4iaAFtAeWO9TbT5v1f60zqkyFnY0djR4B+9dDIuvAvpzB8W3UYxQgQ71oT7Uh/p7jJ6s1qW0NYbiE0MhhgJuwW1jM5Fr+oN8kfx6Uxsnal6ALbAFtt2uG1+yavVE3cNtxuRRN07dODgH512ootnfON2NKu7dIn3oYVhjWENiSNxgEj9U3T0/gTdEOXYc2IC/8Bf+wt9WRpG/ZObL5lJFJiWixECJAdSBequVGDVGMnwXSxpLGugC3aZCd11A+enYrS184QFdoAt0gW7TCwmXgxgbSgi3qx+svfNd2yfXAl7AC3g7AN7VRu+WANbx5QYz2COSTCQZzIN5OiIFqWZ16+MZJAdBOkgH6VjuT7Tc15vttRRr+33CJYRLgC7QbUVHpIOsHmRYCwyGwTCYor+GhK9pAg2P4TE8br5NfNoLenHvec3iTY2N9m4am+1G8TjJ5Z/DQ4g5v07yhzHmU5VfKxXfddlY7KMR6Jupw8xgHIyD8RUVhGWLJLGqbz8+10DwdfEMSgjpg0QfJNgNu3c3v/bw+/RT0EJQBCJD5A4mChvJY+pc4DE8hscdKzCsTwFNVhDgAlyA256GoLryZAcCjTpnxLqUoVCGAufhfJcbjz4Sgm7hGCuiGRjXQBfoNi6aocdW3Yczbj8dYMOkPpoNNBswHIYfakuO249lTw4N7121d64vKm1Db+gNvaE3XaLb2JzfGxD4IPABdsFuK2UcOsPY2siHT5tSkongHbxTx7KbOpbNkw33XktOBQumNiyGxR2sYGlUTw/a3cFhOAyH2ySwe9hoabPG7qJ30xstquw2Sqb3qbNz6adEPyW4DbcPyn5eDk4/q+W8mdx7r/r2sZ2xnWEwDG5wEeJqkd3z1yEuhZeN/eo22t6YNJsGIzlFh2BJv5qbxsZ1IP/ST6NwdgEDQ39LpZ8puErC8T1WADgAB+D3AH8b3Fzq23smkNU28nt966/0phrvMmPo+y/1yZVtqzZwJCay3q8Zk/Qalmv6y9sfyhMulK4phlxMww/T6st80L96BL7zl17KtZ0W0UP8lu8Wxpq5s688x/DTbd1fMmFnYFT9WvUxuQ6EYLJHYSwbLufjaPZwwiAhsYaYHGJZZb2XQbXU1XEQAOr/9vaHN9rTmeuKgyKfJGkop2D2PnJTNNFKoI+UjmldygPegEJQCAofxBOWorzz6yIkdHy/Z3wrb57eA0/fu/KMywdM9bvLDRtVNmwjFMb+EFSCSlAJKvdrNdoOZiMshIWwsAssfGWcFjdyqiY6BKWvnz561X17gD3ZydFITWXdlm7V/NrqD9HLqysrQjlMu04oOc0n4hZ1Do3BInUMQBfothW6EBcbFByCw9aGLgWFV/qO6TOpjQh9u6p7qh95HsH8WRsbeg917rV8PvV5THPhtoV5k6Yzw0/4CT/h5/PzsxzJWSHz+vaj3MlQt5AJxrJob4MbrSe51CdNdk5LPUsnTakZWH1f368i1vUAM9A2oD3YuvYyTYLlFi3AmuPe0+IL6kLdZ0+4F71pL5zf6ydQ17Jd8/ixgcgg97EyqubwljIpKAtl9xMbGD41NlALb8npEw+AmTBzV/GA249rxKD2V6tB6+GkSSiAUADABbgd097Xn7EaYKFioQJMgLkTYM4SVl8YOXU25avqCZz6YBJMgkkw2dRCpSpLtZ9SJc8mjUQaCZJC0sP30OuJh+KVY24CSSDZzlx7/VFN3HX4CT/h536NzGEbWzD1QSWoBJWgspUlnxtb4O/YyKR5HeSEnJDz+XNCtx8fJIVuPy1mhVzTNGlfh4QTCSdYBav7UCTZ7rIiSRO4bkkS5ifmJ5yEk7U47vaTPffa+o4Q8QScgBNwNsrA3NiiiSohkAkyQSbIXESmZW5AZp0ZdWAJLIElsNx7XqiJU436pIVIC0FVqNqN1sr2khVaozqpDaM1AS/gBbz4/s/dW7n2eKnvEQIgBAAzYWaDmTlsxBQQWiSDSlAJKlskY6rGHzfJ3LQYlMygZDANprtt0TYn+IoEAKMWWkLLJuullmuY6jdiqWeCmTATZjYrwb+xF32NKlMPYRX5fbgLdw+5tb2OtO5ZsOq5xFKJpUJciHto3Um1v9/SKXWIBIgNQEyI+fzEfGWcFjdyqiZBZmT6+umjV923B+ao7ORopMQDDJdulVwQo7JykxlWQzlKPfSl2TQYyWa3wUR9NR+pZFzrYzCNwtk9CAz9Lcp/FIOrJBzf3244Ckfh6JaWp93i+chMEsHyhJgQc1+WZzXHc2vbc/+mp8PsT2Z/AlNgeqCtnZ1mlEWZ2J3YnaASVNZRFlWlito6l4n+zqATdILOpgmayrrRvXfg8xwETQiaIC7EPTji6q5PzMHD+oSFsBDHfU7Eto5lsgAn4AScgHOv2qSKmF+jTaK5M6yElbDywB1uHbVs+IwRnwgnEU6AC3C7pVxyN/RzrrMzCc3vME5hJaxsEisftnFabn4nMDmqbvs+nXgHTsJJOAkn95s11+hrmgtPmSUshIWwsMn+9bLJWKfAnelywBJYAst6YKkTQtvwsmkzOckSkSUCzIC5vS3sbz8+e4Ol2uSeHvPp6V8PdIFuJ9JNNmNDiB1AS2jZUZX9Q/u06nK3lYVaJyxdet3R6w7QAtrDyu7XMVZpSPSV6CvEhbgHWnlfWrjtLLxHh0pIAG7CzWfnplim1YUrl2Z+PeUvyvtZHu1e3ptW7e/lMpUvm07kKsminp3JeZTnDlO53LKclmnIVS6EiTV1uE/y6YfyJDwCwvuDs/RyY/YWd9S7R2N1kJ7D23+VXWTlwsr+htNwtq5yAUZRUvrUsrZHudxfYU+Wld618dP7t5kgKCmisRFrR0Lc8gpJ4+rAahrITZSDc67Zp0kz3/1MnyAxGHHuQSbIfK4oanUGq6fR17PKPP21CNPKHhErU0zFyrVLgzibGy8/y8OVP3WOxuF5KIdPttwonUB5h6yYVodS3+JePQ1KmsDQzbZkXQSFg3AQDq7OJsnfyAror9Uz3is5X/qX6jfZ3zycBTkrQzKU5xqHQa7kAMutLU+A9rJ3Tbs+FiMWI6SElI10srUFuaWLrTND+/GyZbGbC80tMkG1cZPED2yFrfth60/aCNVXUa6rGmfVCmmcyv2q1EjyXsFsdGda7eLcaJ0kkVzYnvFOTeUglwtXZePHatfmqIvzjfMN9sBeE0zKfWVtBkAP6AE9oLeniGOZl6kp3ujAOlgH62Dd1xp4t5/kqN9+bJmJ1+CI4Ral4rVFDKkMh5/ws3kqnUHtKh0ChdiRcBAOPlNbooZ7zS7mI+Yj2ASbdWNzdtpld+VrHM+O+b2KUR/E/yqyfF5cNjayRK5XqeDI8mSqD0MRz4rSyu3T5qV45RTGIHOEmTCzU8zUFuUGYmpFY63QHCJzROYIW2Frm8KZs3Ka1pUdmlioWKhQFIo2zEKddVxrpluPhYqFClth62GJMzVy68o0+RAVokJUiNr60sa54Up1I3AFrsAVuH41XKuYarMrx8nwEz8FmkCzNh8/D36b91XXQ4GqbuvlAddTgQw9b11vVBBGZRBgi+bpvXoGqt0pOj/oXvCPcFNfMDkmSZEtvnoBkLptetXW/MNpED5Dy/TXoexpvgKYZRD6jgTX6tQIxuNUtq/69+oyuDHCyzAPck2yPC0yzTvZ/atwpDQLS2IlUw0/+SYzqWgRywPI591fN0xG6Af9Fuj383zzAjE48mNxsM71CV2E3MZJaPruTvUcILnGc0NSrm25rvqJd0vBoQcFoSAUhILPYANWxlyZv8n0fEjtD1czd/WmVVN39ajdohy6u4UJWA1qyUYTNS4iJZschbLqO7cM3YYycYsoZf1gJF4JdsHuvgecXcuOBUap4pSFu0oKYWYqh7zsBhfmi9boSI3FEtUO+ZG8aHSxbI2eqjO9IpfhOA7PJ7tOuHsDzE/MTzgIB5/GwV/iIitkpaLkXNaj2g09YvedugrV9SxJk4dXpZ++t+hjtdBr0DcYgj7QB/pA37N53uJ2a+d7a897Fm48mu//MgQrltbigQ/xwPHAwS/4PVjLcwv07loyZGN/Yn8CQAC4B/VPL+jFvZUCoKI37YVH1d8uQXDXkUenofh7TC7ZAPsS4SToBJ27th21K7/OfNyYw9m1cz7AOcc5B7AA9tBt0/rE6Y11zrFOgSfw7GBiafus0mZ9US35JN8i1kmsEySCxPoqHXfusqM2goAQEAI2Md1da7zSBX2gD/SBvvqMv/rlPlTaQEEoCAX30O7ikYLu+ppdeIQBYSAMhIG7yYxowU5rS24amyJB1QN+wS/4fVrTi7Uy8303vPAQ9iDsgZ/ws3U5nI1dLHedwyF9jecO+kBfDabjNi77no1ISrehITSEhk+l4e0nHci8/fjFGu9iutqZpltQvOoBGu1W/1Ge8qh8zDs3unxxIO8g/yjK6UnDaV453kVWTZS/c7K1fw1X4SpcfaKVuSlpvgPrcrNYnKEQ2JdwEA4+k2JSJ8abK5rciMIhaXHS4tAW2h62Pr3W3A7KdKxN+Af/aupX8eh0xh1FNDfbnaR4gCJQBIp71fpoR70RJqGP143XDWAB7CFPYKwaTzZgCCOt0NGkg1EwWrsUKb/92NL2Gn4fDRIaJIAKUA+j9fnOvf92CpCwSAEoAO1O5PT2Y2REt5+aNFR8kyNvYoRihMJQGNr67NOaAqNde/EmeSfyTqAVtB5CH85HB/HsuBNnO4X1CJzgIBw8JA5u2XOzxr7EfVgIC2EhLKxHAU9TYsAIGAHjwYs0dYSx+Z3f2jnIkRAl1IW6UPfp5mijmsvRaxMTFBgCw3oF7vUOD29srgZdETAFphRdbuPMb27zsW83fmhiVmJWQkJIWI+P/ciM3hr8bHoOA0SACBDrGW5Wj96cgeRAD+gBvdpqFetvnNHY5kPEFmEpLEVQvkJQvn6YeM1FNQOXvhn0zYCf8LPmiKSKZA9kJfQmypvFsw28DuWInSq9ObOflhbJOLi5j1gmejULZZzq14byuONQbnd0c7xN7LKuxkRzMsojyPrmH6YaUUWqHkFtuQ5Fqq/Ph+rajmZQvGPumndcwPHT/fkTfUFUJkzLJvrSj1UwLm+jbHN1e1Sczamqfxipc00PuTLqr4U8g8pm//KVj3i/a8bs6XvGn8uc2wNSzZJu+ijG53AUjsLRfXJ0TcyznlpHCApBISgEfXbJZVacVkEwvW/yIOJpZnp9LNO4DOMi177f29uPN4uMLD3PqyQcG1FSps3FmZcPPN5OTLRz7ZDTToRuEVFtBkeJqsJiWPysLH6dVK+Nk+u7sFx5tIL4RiyrPxj/VejwaRrE2ZkcvneZ4fi+vlbybMJVucCLaKpJi4SNio0KF+HiM3Lxj69+fm+cfP/qzcm3/7d03YV8tluugrZR3bmNqs9uuXrz+zkPAUyVLFtcneWRHFQVRWq8az4O4SN8hI/w8QCioFtIoHbu0BMThafwFJ7uiadbhER1p6S3QbOCopsGBrfUIt2iaVJDQqI0TgLSQLqOoIDOTZXx0BVRgdtPTQoL+OSlyEuBWlDbItSW9zRYCijoCir53/6dNWwUcaQl5/pYzlNU2kAWKlu2a5ZJrXq60GHfYt8CXaDbOujqYlX9/refvgy7mrmuaT4bczer/TFoMWhhK2w9wABv46K7Byl5JUMGQAFou5Sqt59WSFXFXm2oVnWIjYqNCmJBbAeCrnPW7iHgisGJwQkNoWHDSwB0GqqmdL9vkpMiJwWFoXDnbdIhNik2KTSEhtDwmTVRB5mfB4gAESCSVt9f+ekBqvaBKBAFot2xKu39JH6GA3AIDsEhOKytJ57O6SA1IuFDwgf6Qt/mdCTVdUeazTXh16cGFPyCX/BLDei+VKBDCkAR1wNWwErH1IWOqSM1DtKj0yC+OBpN1Ohin33+6MxPdBaOwtH2Zfx15KBJPaeJJhBNANbAupNGbxXGbcy81IOcOQ2pITWkhtRPTbvZSB6INwBGwIgJ+7S4LfNZ4CgchaMYmPln1fs12ZdwES7CRbjYSC6Wdfx1iV1NwAgYASNgbHZV6nPOItkUh0T3BA7BITisGYc6b96Y6UzkxcmLg12wS/rnQfpHU3rPCqZNtqsFp+E0nIbTHSgLWDCQv7wX4DRIRyo6OlNquSBA7tSOEe2CaBANokH0ISD6vZy6i6x8Sbk4s4uqH7bK9veM97Jx8ixK15bLuwZXQRgF+nbKApZ/M6vQMsZBNjlNgvRZhqnIgq8B8BoT+VTFckbX0PYpkdc3id63RHiyvIjGdaAXThwEvRKLKy6HXv7Zuj7Szy1LlxWRLE6ishIs+g4I6+52R/4sC87U42ASLswiTTozqeToCwYSIUl5teZUnsOn3JkiK+QWfMYg+cAruWqnSXJxGaQXdynOVP/DK+9fxLIIlec0f0q57hcZKAElj1h7sk+hrMpNeaxHEzUuItkJ/Wc/JnHphMt1mkNGXpOK4SAW4Vj/01V+kiDiXO0WIA4AASAApHEAeacutW2cHldrc5mchtFco50ZOmAnOznHyC9ZtY36SGpzOdZLFsnxK/QvEuMyuNCXZO547BQoPkABKACl6UAJ5Z3SWOUPkZLLDqSplvo1CSp9oAJUgEoz3Rw5YongZO7FyPudhenlzNOxPHFyrtSl7PRylHAkRyAcyXHVwUn9g7jQL9JncSlWeK1OszCXO1NFAvUlKMODcgp3HFqxgQ7QATpti61Y9gJx6g+wEKGFIlCk8f7QdSD+0ENnaAkl+ENABagAla/PIOu6yAamkIfABJgAk2b6OZfyBhM5RbMEkNx4OQ0zd2YJHaXyauFg9gx9SLQMLFVXobo2QlnAiYplzVIdo7mS671zO8XyYAtsgS3oU4jCQhAIcrjxk8UKtIdRFK+hURS0b6AFtLQtiqLbOjcwigJMgAkwabOd0kzpmwlX4ApcaWf+uKmej0UCGapAlba5Po30fFxQAkpASRu1+XZbtfkEW2AOzGm4ZGUx1NIy3coAvsAX+NI22cp9BLcBwhUCtzAEhrTRL7rHSKucIpRyAAfgtLSVUzOTz+SJYApModAQmAATYNJlKUsTW01a9FYBKACl4Wmgu06TLcsBYa0AF+DSxvjt3FZB0gJuwA242VedM93iQAtoAS1f5SZVURd0cpAFskCW5/aRyu//XapUXO7wa/lZEH3uOd3XK2qpnH1sucbbH8WR0vM1l52mhX1Pv1nhP81NnL34TvvG0PvFfdAPHoRxZshlz/XihnF16OTlekFm81f13t+BRwsSlRweuXQ/vX/70nj7wxs9XjXL9Ktf3mXzZAdliYOonLpannpxZtMgzs60iQnYAFv38t5qfnlk8ec3Rp9/lZdGQBiHuZ6yPL6bHTwOx0t3Lsw/P9HfPBhOXH2f0qyaJ8nnx+awokCQDJJBstpMtPkxnN+id9/+i9cfDhzbmAjLTsVcM4rpWONMG1TfyGZNg1yXPHzTM/TBqLyd6gg/dBD1X5QE07+Q3RZfMjusUSfAC3gBrybBq+8M+vZgNbzksaaR7FtD2QW8gBfw6gy81G/yyzy8FLboAPsdsabFqTz5RJ+6m0o4kESR0nHkk4kaXVRHOC/0HhjTJNVRtEpTUK7zg/MKsSAWxIJYz0KsKDxNA131LodgXY/BdyquMoGLhtX8D2e4StLq8s6EUfPfjlV2cViNwuAVvIJXzUk/vtNdOzLZKeN7FUTygxP5Vao29iXTaUj/uG8ar1qQhuwDOAAH4Dpb0p8Vp9koDaflPqbaFMsWS+d+lOc8V9UvZJUzlef6a83FXbMTOyfXQ9HoYXWZB12gC3Q1xzY70dt5ojf/pCqUORRpGDYZYANsnQHbQu8C/dq5eEs+Wl803/Lcqh9kMJalfF0oQ+cnj5eDaH+eqekfiu5/VHmalNL7nRLLh1gQC2J1hVhyR4RYP717Peh73sC7z0Rmk3A61T1rdVFLZtwdMNldZ4FW79NgdFHhqoytlUd7kkyn5T2V83s9UWnlTsp3HJW7rKpPJT8JyAAZINtljXSLY2Jo8eEX/OqOIuxhZ4cVDqRlOt7XO5B/KsLRxYnszO6dSOL5sAt2da6JxOb2EW+SNJ/IS9Kqf0R5Z/Xr4gcXNJ2Zc3+Q47lw+O+mqJze3HVXQDAGxIAYENsfxO6j8M0HGKF8AAbAOgOwcx26X+M52kPL/3rP8USNZ1bbTnnlwSt4Ba9Qga1VgT2Q5rdGBkY8DLJBti63pvB9u++Zq1tTyC6O5K3lm9CbAnpBL+hVF73ezm5GObdPztexHufn+f79pL/TNAnGp/qBZqdVV31XQ/5mtpfAzu4PXWsw8yfzxTmAy6P/iN+DLbAFtvYYv78PZTU/fj8EYAAMgHVOiu+5tu07W0nxvTZI8THE4Bgc66z/aNmLDuSSOn+9C+k6rtOv3YVEQQG5IFfXexzOq4Ua3d4QVIEqUNXp7vem2e+77RzdYUMv6AW9uuoi9p0vTjH67tCz6/YPkUbALbjVpRxj2Wl1N/XZe2rwRTQeZIGs7mUVbc8ZOMPDySqiqYdjcKzTmnp32Pf7bRz3OIBdsAt2da83YdUgek1XwiUvscmtCYnWgy/wxbDah8Nq9QvD6hmrA9rEubXAC3gBr87PrbXbMrcWXxFewavudsX5LhVbq9zh1/IzWcjPu+K0dDga6gnQBto6HcIfuK5reoTwYRfsgl1tMsvehbJIpSLiexVEunuh/CpVj1pnlnVsmsarFphnEA7CQbjOavJta7FsWzeX3iDH7/dty3PrluMzbRtiQayuh/a9tkT2CX+BK3BF0ePWRY+O7Xpm/UWPYAtsga0OR+2tgeu5Thuj9qi/gBfw6vKMIR2ll/V7HQbnsex+OMo+D9k7LY3YY5kBN+DWacvM6w8cu5U9wAjeAy/g1Z2iooU+0Ov66fiW567up2M3p58O3IJbcAuPcqNHuawB84/77XAo6bED22Bbp3vsOK7V0qbSGGbAC3gx/3Fp/uOfinB0cSKrywxIIAbEgFhDu7VaQ89x7cPp1ooxBsfgWAe7Tjt9d9F/3MAx220DyFCQQTJI1sXerQsS/dX9W5cA1uT+rTAMhsGwLjKsLONeTa+2NJ+mLQXsgl30Q1zfD7Ga162FFo5W7r9FuQ/UgBpQa7KITG/nid78E9nncLSijZjd0qIkNGSgDbR1MPJvDweut1Xk32lF4B+QATJA1mkx7MBzLXO1GFZ2cSRvLd+kmY0voBf0gl54mOs9zIX0ZauKlJDIQjbIRhvFqgjgNIzUph6K3tBxnH7dPRQdmAWzYFZXrDEtu1jX5MIyHW9lkwtvix4Xy+VMiMZgFsyCWbuKf9n9genb7Yx/obCAXtCry/Ry+qbTb2dvRKL30At6dZpertf3rX4be+67sAt2wa7uFRsttnhtedER8XoYBsMoOlpfdPRAPdGayiPIBtkgG5lIuWYDx7RXZiKXGlvU3G6fSBi8gle02/+CdvutscVQssI22NbBSNlctro6SLY07ajJUTIUYvALfnVWie/6C0r8peD/ejn+wPPMvlW3HB/DC3ABru71qPBNZzjwt+pRsWSENbZJBRYYIANknQHZfUvqdTF9e2j5q0foNqy8iAEhkAtyobBYr7CwvJZ2QKT0CLSBtu6i7V0oi1T6id+rIJrnMNVjaUv72HLb0bwaTQaAA3CdAdy1OJDpl6vItnE43yRpPhEopbt3OJkjArSAVmeTldZisrIci7ShZ9jA8Ya19wyjjhJgASzcyEfdyLY2csWNBHAAroPy18qhbP1UXfAFvsBXV+yzRYHrTmJhb+XhldpDJIz4PeSCXNRUbq6pvJ+qi/ICsAE2wNaKpq8D1/H9VjZ9JUEJvIBX94qSPNuxPOdwBud6cAyOwbHOccxxhv2heTgcI0oGx+BYB9OTpTqs5a15mH8EvaBXB60wq2/57iG1uKBXDyADZN0Dme1anklYDI7BMTh2GLlJs+8PzHYOpARewAt4UYv0WC2S085O/Bhn8A2+dbAhrDNwHHcrJ9Nrg5NJV0U4Bse67GT2Ta9v2audzLIsqXzXURJF1W430t30wRgYA2Pdk5B5pu3aW5ljS2XijbXHiJsBMkDWZXvM9v1+32pl0B8rDHgBr87AazaUcl1vC8t0vFZMFiGWD7bAFn15FtjlW57bhh7VqPZhF+zqcmcevZ0nevNPZJ/D0aNTkVrT7JXuFpANsnWwmnJpMPiaqkqvJVWV1CIBMSDWQYiJkabSWOXrCOa0BGD0tABgAIzJbZtnhTtbRMV+VHmaEBGDWBALYu1wbJu9OLft3gxbP7zNtQamb9U9vI1wF9gCW50Wfg2dft9dLfySx5pGsm8NLfa2gRfwAl6dzUJ+lwqwyh1+LT+ThTyI4SCYZFANqqEL20rTajcnBEbSEW7BLTRh6zVh99aYfWy5rWi4gzEG1IAaLuZ6F7OlncQYrgvZIFsnW4nZtusx/giOwTE41mKt66wOfI3StS1SfRxM8AW+OmOG6ZltX1z97TSr+huLC2SBrO55jgPPHA6363potaILNaJ9SAbJGJ275DhWScpmO40IX+EW3Opg3+lh3zssCwyQATJA1j0D7DRNgvGp/vSWx+9dCAbBIFhX675ta7Huu+pjsaHo2/JN2xzUXfRN4STQAlqMyX18TO69Yr81wlYEFbANtnUvNuZalj/obxUbm4f4GY8Lw2AYDGuyKGzgmPZKUdjSYMl1qrC38tRK7UEThtUFsSBWZ4h11+BwHbZcr29+fccKRhlBLsgFufY/PXIttzwa7UAsiAWxGtZqemls0YbEo+mYVr/2xCPGFugCXZ1Fl/dFYgnfGgx9G7EEzIJZMKuBGtWlWFaTRarYXTAMhnWxZNsSG8o6DFEEDINhMKy7Ya9F37Gs317vOTqWY5nD2j1HGuVALIjVxTLtgTUcbFembbehTJsQGCADZN0D2dD3Hc8+nF7RWGSADJB1etzt0HS9vr963K1s1jTIR5OGzrsFXsALeJGIXJGIbEu3HBxJEAbCKBOay+3toeWvLhPaRm//pyIcXZzItuxec+8DLsAFuLrXVULL5r0D6riKBQbIAFlnVRTuoopiwa9cr6UY+P5g6KClgFtwC27VGLl3zb7rW6sj9/JY00j2rZmBe+T3wAt4EfZ6fNLjVk299tRmgiaEQAtodbBmyLV9d7tGqkS84Bgcg2ONbnZf9raX9XsdBuex7H44yj5vdL8QvG9Vt3u6U0A36NZFK21gmV7/gBKTKCwAGSDrcoDf882+6awO8JeRsvJdR0kUVbvdyFi/A8bAGBhDX7FNh0Pbsh2n9g6HWF4gC2R1sLrbMS13cDjV3UjE4Bgc6wzHFjvff/kYom3qi06Cy2mRnSRxrHcbwQX8gl/wa5cSV3toC7Da2JyCoUTAC3h1z4m0+wN7sZ9OmztMuzAMhsGwDhtg8m181215CpJqIzAGxjqtpPC8gemtxpjs4kjeWr4JTQ6BF/ACXu2dENk3Tbv+OR/oJ0AWyOqyvTXw+745bKW9RWMd6AW9MLi2Gas2tD2779VtbgEsgAWwOjhVzR9YzuBw+lKQbYRjcKzDfSn0dp7ozT+RfQ5Hh9OUAgsNskG2LgfE+p5tWoNWClk94AW8gFdX4HXfA383NURvkjSfCJjS3dcPIcGHXJCrM+S6lDeYyCkSdsgx0JdC/jq4EgspOI3UHESf1zAa5b3Vr40fXNJ0NnbyD3JEFy6AvqZhqk/xzfzpEFQAMkAGyJ550O1M8LV6yG1bZtwS/YJe0Iu4/tZxffvYconrQzbIBtna1mDHMh1vZXBsXuzdkN46yFxhF+xiCsjmKSAL49daZZgR+QduwA24fQncLEvLyV7hdkI36AbdGij49z1v4DuH05+avjxwDI51WRZrDzxvXV+ehstigRfwAl7MNVrWzG7qzTMYesPae/NYcAtuwa3Ocsuyvxhcjm3a/drBxVxvuAW34Nb23BLX0rYHtRtcROsBF+DqLLj6zpdyyx463tDG3gJbYAts1SWh+C5VKi53+LX8TBbyM/3EsmjfP+6jngBsgA2wdUe0/6ciHF2cyKbsXrBPDTjYAlvdE325A3M4HByO6AtpPhyDYx0Ur5p+33S34phlAzJABsgAWYNAFsrfp7HKv9yJ3Kot4oka76MlokVYH2pBLWSrVcNEYZnalIp0B0PPqT0ViaUFs2BW91xGa9i3XftwQl8OHINjcKxzHPMc1+5bBxT6IhcJyAAZI0GeLqDY4zQQxksCLaDV5W4TntUfuv7qbhMlucp3HSVRVO12I/tOYHuBMTDWZYxZVt80zZZjjJg+GANjXcbYYGgvCcJaibEBGANjYKzDGHOHA6dvr8aYPNY0kn1raAdDKiGBF/DqrBbMW9SC3clbNzWmcPoDp/aGOgTAoBbUQnc/zz26Xt9crbu3t9Ddv5VHV2ofynsar8ItuNVpV9F3LXOw2lWUXRzJW8s3Ic4FvIAX8Gp5E0N/4PlO3b6iB7bAFtjqKrZsazHEdS3GVrqJWQPXcT2vbmb1YRbMglld1qkOLXdotXIqGrIu4AW8SCnq5ToXfm2wtixTjC3GCgEsgAWw9l+T7Q/sgX1ANdmUNwIyQEaEfmsxl+l7louYC2pBLahVY7DL9hzTa6N+HtfxMNn1vzvvZBL4BggA"
EVAL_B64 = "H4sIAH405WoC/+2d3XLjthWA7zvTd0B9s+mMzJUs2/rplWtvErfZtWt7k8nEGQ1EQiIqCmAJUIqyszO978vsE+wz+Bn6JD0HICVq4790ZUten0km41AUCZyfTwcHB8C7rbEwhg+F2eqyn95tZToR8NeWmRkrxls1thVqZYWyePFHnTOeCfZNrI09l0PFE3ZwXGNcMT0YJFIJFs76IjMizDNpZ8yEfMwiYUVopVZsrCORBOwAvjf7VTAbC5ZmeiIjEbGiGbXidhZJM8ylgU9snAluTY39KjK9PZWRjV/GeqyHySyNme4PcngNPh9uCTMRQWMlNAyePrDYtgjabOVAhng1z4ZChbOAneQ2zS2b6TxjE5FFEt4pFTM2w7/+dn7yhg10NuaWTaWN2UjMTJdl0ox6iZiIhH313ckPNfb61dHx29c19u3xN9/+ueY/N6EGGX1V327U63ANZdCzsxS6Bo3rDRI+NOyrRBoLQnPvU0MD91U6UtwufkkTrtwV349MhHo8Fgrk1eNOpMHW+xpbaC03IvtEZwthS1MKGXvmdSMVdJxbnZnupbrcwn9+EE7Hg0yP2aGIeMYOtU5FBu2YCNYPeKCCETN5murMBuxHFCAPQ50ry5QQkYFmboNEUeKu7QE70kxpUKlI+KzLYmtT0335svjSdmktgfiFj9NEBFYY+zLRQ1m0aLmP3BiQHYfeLXf03aVi7HJroaJLuHq5hZq53KpVPnT6wQ/breLyXEX+K32uRj05hj4b7cW/eECpQbzzJ7zGbvsCfrgwyV4m/pVD33o66xWd73njrtxeGCj0Gy78XLz2U9vw7Yy0tSKat61iLv7zw4TLsWFWg0bSTBhoBeMMG+usCS+ZHP7jHBFsS6YSb4H7nfpmDKwEPrZozuDipZKhyTofxvCo0pzA80cB+95/qfwYH4rvemEQDt77eJriQ1FrYL8sjLlSSIQjbQ2bCniPVGDC0CvWF3YqhGKJgE5mrhclEuDN4I5TnUVBRTGfuoaXQWF6YSLDUY2lfFbDBpgYTXyhGROAkfs2YbN1NuRK/upEOe+PtL/tCEgFPEik+F5lk5kXnITmz7t2PEDGgL+YFLE2kIor9whHSnAznkfALZ4krNFp1gs/R99iwB53U5jJsQiGehJIFaBhvN96//P7P/7hHbGb2L3E7rdn37x6c9H1UH4DNhQDKTP2V+fyhftOJZhaH3wz0eEIVaQjPpt7799/PES7Kxmdp3I7E4NcRUt0ZhK6EkluRTILHhjSzacNaaWV+AIQTaAl0BJorw2ST+EVAizsSA6lBek52q4gPg4x+N5G/9wOYxGOlgj8wNBtE3QJugRdgu4mQfdrqa4+JGzKMwUtA+O6+nhdigL522V5GkF46uJZFBMIcIfFIFLjPERkExmCh4/5DGNhUCnvJyIKNgG+ncbThm+i9YgncvQlEJh9B53Zdr2ZZyLcjz4H7sYCbTWTqfW5i9wxoZKnwBSFIYoTxYniS6Hz1QcXO9+M71WEz6pMf/i7Z48ZPu8TwYngRHAi+Jcah/NqGH5NDqSYJpyH4aNgFoT/VyC+Tozv0vwgzQ8SuYncz4jcK8meLEGbPS61O098wtCkPATtfAnYPig5DCZnmEkTWXgcZ9hLgW3iEy2jBUeI2ERsIvaC2Ev5D1e6Aa8VmCc5M2xnt9MJ2Ct4euZF9/b0mJ0ev2HOyOENKT4enAxaMJDZeJEcSXkWimR7IMRy1R2403pSI3kqe/ByEKft4Ud3sLm8dQxenebJp3R2T5MKkRzqJAELLin9+bOHbw2glTNfFYNWBD+nEaJDKrAHMJ/tonGAKAA1ih9sHLSDuuBORYW1AB9LhYGuytwDz22sMwlGUjwHHAmB53jvVT+GBs6IlERKIuWClKeF1w24dLHoBfrLBJWKqARS1tByQW1egQUXWMEFlKGTaR6kgXQ2rvQ0WE8B8ibB8K4kwsbgkJIEBFIC6WpCThbzCXoI2iRmCtAvvYdjk+eR53nIlVMijhRdA8VSLPrff//HuXPFV6Xt3lVcTAGoVBNpZD95ClEoOy4bi7jK4KEI2mUMSwVfQzux4hfwyGksw5ih7cTg+W6ObZ6OMhLtAAiX4HMIzYRmQvPvQ/O9yPwJlhHUVTKvLz3QIjoTnYnOROdnHTjfFjVvYGUyJW8peUvoJHQ+LDr36vX656GTZr0InAROAueXVh9QXQ7BdGCDFF0LfV9m3r0AmkA+P8rLuDKlL55D89zV/e0IC8BYCEpnbjwITyhWYTBohLLBesq2tE17zjjuYGOlIKt6OyseMQfhQxTSHpjRNdVW4BNhot1ME5Br24JLA46McXNO7OTiFAQc6zyJmMJ5Sqykc5SKvA0jIMA5a176rvCuNAiDRgXjb5ryIooSRVdAUW/tLgs6VNCTbmHpqC1wVGeJ/8yNLWORiBkNfueyZMbqFK0hV0UM4/SHkB2I7BlD8x5p0rVxk7KixFZi6yOx9eqjM/cFWd3y341Ha2dz0XqP9blrQysttyWAEkBXCdATXAGADlfsf+AkhOj0K3qYX53DnehYGbWWSwZinYBfBuxMpGDJTnK++jUSDw3Q9uYC9M41VmvDJy2ZIl4SLz9vE1yVmxw3CkuuPg5BIl4fuCTgTEykmBZ8tHKCGq5sdnvrzPoD07J1Ay3noV4P57vuQCb6F1iJzk317gobcWrIT930+lzaFcWaRxL0aq/hpQvd5zSYij7jUYTLU/3vFS4flmNpcaUxh7geRgEY/vvVxfNtfXSK2UzoThFI5goa4BRaWgNFnURRouimUPTWDVtWQVEv7Js4ukscJY4SR4mjG8LR81J5HMZ+tssM6Bklc49DFsC3UlzCivt7FMN53LcDJeqHfQ9asrS3oSC9vU5p/Qwl/hH/iH+LAqWrjzmLrz5MMD4A7LGo2GhqonMYZWdg5q5gUNoqEm+p1GR9MUCJjGWk5DB+8LqkOgWUFFASUAmoGwLU78QQh+X4/KuPqjzOC4fo8C8Y11iqHFgJ7pigQ6NhllPmWEXf6nQeYWeT1k0F7wXuykr2crO6O/iJEy++ltz0qsWuFZDe8MQH2Pkard0Vw5sYPTgSPHKuBTrzriCUKWGJFxMxRBSA/eOUlcKNE31k79q5OEqtrHyfb7m3jJ3iZw/1roYEVgIrgXX1kWrGBK6bAVGgFuFpqtBgeWqiI5G76o9OvPoAIj3lM2/6KM9csD7eXDki8bFWIP1fmVFCMiGZkExI/gJjXVxsv6pg93a27hBbia3EVmLrE2LrwfkFO/z24M3hqz+5CBaX2Je7/Tf25mzVxdr60sXLSDgVIDXlTdkvL8VtVx84rdB6mpy94zzajUAs4ZHwSHhcXlgPepIglRkupzdhLKIcGOe+ZkHyWQahJXujSzdbWnTPbbGG3sKbVoFFEPQNVGzcdGCsUGCcNzDwc4AFXQaFaTX8RHquZn2gccsWqZZEjYXtiZ5uY8OxiCFPwLQ1cAuJ4jflXpw7glLjA3E3kQAI/n/xK7g4QoD/u2kp9KkSmCV1/PbeJgfz/w184IW49KwPUeaYZ6P5HFcmgERaXTO35We+iCHEkDsYMoYHxGBFBqdTS0zwCZcJHprkHUXMj2ECgwzYBb4A78LfRC86+Jjj2R9j/JHM/gLGWTH9OXT6s7nDPSxx6o9MnIuqyN05QxJCIvBri56yLKRybTaqec6Ycpt/EOnJxWkNtx+qzdfu1OYE9ycE8MSFNs7A7WLBKTGMGPbsGHZWIKfrZSNxszUlLEThfZ9zQU1mQompQSNrQzw0EeM+ustraO1Q+A9B1kZYHGiYEnmF3U5F30gYq2Bor1DECZgrtPxhCdYigBHACGDP6Fygcn+JLia6Gp2O+9LMR2h9mYhFfdsZ3plapnLkGDt79XWrVd9r1wP2fbnwJStuKU8NKuqziVnELGIWMWslA0fwEaDPydnR/n6z0e6wmBvWx639TCzTFFNMmGcxbG5goN3XWkUcJHuR8XCENdXgsbhflTPrWKep81Gw3WmMM2UYc0H/QhEVydpoNfvq3MwwGjgSw4hhzzbuau5W4q4hN7cFXfvtvZ16e91B1w4Bi4BFwHo2R22nqZbKOhdx/T+EvzKQ35HkQwXal6FLzRcnahezgY1F2gsn/hqNbr3ODl4H7BAno1HdBda0qio/e1GZBy/zYFazMR+5nUmVX09K4RjRjehGdFtlHr+faR718e03JPLLkeQmJ/HbRC+iF9Hr2SXEmu3d5l7jXgmxnb3KhCRlxQhkBDIC2bpBVpph6UVnr75uANPa+wuk5WnE/RJT9gKUlXKL5acvAnZe7B9VmrAjVNV2cVfoMmMG2uYyMVQMRvQiehG9NqKWYnd3r9NprjutT0NHYhYx69kyq925/zxko7XfaXXWDaxHX+RDxCJiEbE2k1iLKvxbalbbjU5zn2pWiVpELaLWxi0UWkrM00ohIhgRjAj29LNbzXan3Vz/YJGYRcwiZj1bZu38jvRWq9FpNdaej98jYhGxiFiU3VquTb1lGnG/0Wg1KdAibBG2CFvrK9xqt+s7+83rC7dAiyE8GnqykYVblNkieBG8ng28qiebuBNK8BvlwaLQAHS3TqO953cU5BEI9CgHFAHOutUVjkXM5crolyn2j1yGo0OeORekcgiCF8GL4PUwA8a96oBxCojKbh0s1nfb+y3KcRGyCFmErMderrjT3Os0O7RckUBGICOQPT2QzUu5bho17u/Wd64dNZabSNw0YDzk4zQ3h/5AzYcfNBK3iFvELdp88F5jxp1GCwaN6x4zNglZhCxC1nMJtTyVboqzdlqNzrVx1oYl5ynRRdAiaD3bOGun8bvXKu7t7LfWvtVzg/Z6/jK59T+AJTIB5fwAAA=="
TEST_B64 = "H4sIAH405WoC/+2d727bOBLAvx9w78DLl+4Btms7dmLnPvWSbtu7bZtL0l0Um4VBS7TFs0xqSSqutyhw3+9l+gT7DH2Ge5KbISVbbpM028aJ0wwWWDSSLJHz56fhcEi93ZoKa/lY2K099vPbLaNTAf/asnPrxHSrxrYirZxQDg++1jnjRrAnibbuWI4VT9mjZzXGFdOjUSqVYNF8KIwVUW6kmzMb8SmLhRORk1qxqY5F2mCP4Hfz3wRziWCZ0WcyFjErmlErLmextONcWjjjEiO4szX2mzC6PpOxSx4meqrH6TxLmB6OcngM3h8uiYyIobESGgZ3HzlsWwxtdnIkIzyam7FQ0bzBXuYuyx2b69ywM2FiCc+Uilln8F//OH75go20mXLHZtIlbCLmdo8ZaSeDVJyJlH33w8ufauz544Nnr57X2NNnT57+tRbO20iDjL5r1lvNJhxDGQzcPIOuQeMGo5SPLfsuldaB0Pzz1NjCdZWOFJeLN1nKlT8S+mFEpKdToUBeA+5F2th6V2NLreVWmI90thS2tKWQsWdBN1JBx7nTxu6dqtMt/O/V0ZPHL0722GuUzAttXGIdN+zvXE3+95//Mh5FOlcolTRlQ8GGqY4mqCUdcxDrj8LI0Zz98/U+A9ElzmV27+HDSMTc1Idwi3qUiGjSEG/4NEtFwwkQg4Q+xZI7kc4boQ2rveLWgrQ49Ge1a29PFWOnW0ulnMLR0y3UxelWrXLSawRP9pvF4YVSwk+wZQM5zcB0dRD48galzvDKn/EYu+wHeHJphAMjfs2hiwNtBoXgBsGcK5cXJgn9hgO/FI/92BpCO6U6k1YOU7FoXsVGwiX7KZdTC+oAa8mMsNAQxhm215sQHrI5/M97HxiUzCReAtefBdWBacBphzYMfl2qG1qt83ECtyptCNx9stB3eRpvis96YJEIweV4luFNUXFgtCxKuFKIgWdlZ/CQgVMgTDYT8FypwI4dGJVU0C68pxNvwJdniYwSFkGzEmAGm2lwWzVmI6OnzEo0KDaSKd6nUdHfxz4T5HSgmdLQmlRGkxrL+LyGjbQJ8m2pQNtg+0W7sRnajLmSv3lxL/os3aedhSaCa4kMn6tcOg/CldClZfdHCB9mc5sh70ZSceVv4REKneJ5DEDj4Gat/nazAEAGDome5S+KjJyKxlifNaRCz1Hvtt798u7Pf3pLUCeor0D9J+F17D1llelggBkaVSMAv/R3JURsoW31YLihCw1WeA1onM/3FnjPuIlEWh8JsQJ2OD5fL897nbvNc6XVt4Bygi3BlmB7LmwP4RECLOxAjqUD6V0Xcosf1UtDWQVvqsdSrRm9XUIvoZfQS+jdJPR+L1GtM27UYmB4HoH3WJ7F3Amfp0ARgfDaLAFxWu8dwpzJCLx7yueY4wB1chinxo2rJTTWnMNoEXgJvAReAu/GZo3PDXq/KHF8SWbhJtPGPUIuIZeQS8jdVOTizNw+xqRg2hrwAm05E1/BXVXmiEMyYn5rE3ZdmrCjCTuCO8GdEhnn8/2rUxmXop7KMSiuJvQSemn6LmQyhg3eUI1rmcHLG1lD1o0Y5Sq+Qeb22nebubF2EGh+C+H0gXYfh89D4WZCKJYK54NraFWJAXgyuCBG0MRr4jXxeslrj+BXh8/gaRAvwxgUEcxjENkhxLyc2SnaGegNwmDro2khsCdHlrWa6GK5wpSI/7vdbW5AWVt/93xG55kcwMNBoG6Apz5D6PLSKfh1lqcfM9rfTSoEc6TTFGy4ZPXXx76vLBCWs/B2q5WaASVKBRYBSqgXjQNIAa9RC2DloEhUNvfaLOyFPfJ/HT57gYrNvX9pxnOXaCPBTIr7gCsh8jz2I4GDoyk0cE6sJFYSK1dYyRJ+hj6GNok0DOREPy2hCBBssGPM2qESMc7xDYRngj3OS9qiR1bcTboNKk7rtwmgBFACKAH02gH64XcQyYf31Um10mEW7Hz8KSm9mcMzMnwAOBm0YSTNdG8j0rCtzadlqvWEp3JyF5DJfoDG1n1rFyN5n03icHki0LiMzFwY+5d3X4zzcYhP02GEXcLu9Yzxu5cN8W+zjrdPMSrFqARLguXdHOTfIjp7hE5CJ6GT0Lmm4X0xff/hvVfcy5PDEG/+mksT3AuwGYk0FDoarmzpi8fQPH+0U4+xYgCG+bEf7vsGFEUDDBqh3LoLZ3fOR6R22cAbx2fYWJnAr17OilssQPglU/hXGMs/spNzZujBLaJU+7lwgFfdgVcDkaz1s+KoJ5BxovM0ZgqEZLDezYMqDmaMjAD/rAUF+PK40iYs2pWxNHQnpBJSrxephbWDdqEbe4WZo6qQq2iG/86h6UVUEjOrwbl8ybh1OkNTyFURzXjlIW5HwjRuZ3C+Cfi8PLK8LXIS9Yh6RL0F9Y5L5X14D54Dw2YLikbRXHlXRPfhd6xWPPMFjMV0ExYmomSxzdcAwCD3CxC4cwECF9HbAJMGn+EgOhuYjM5t9eoK8HB0HUa/gyGX1zUVdACBeuTO4aB/nSzQMBNDxuMYS0LhUtDM1C8rlg7XU3B418CbCV9JYQ3FopBXZ1iiCd0pYsNcQQPgeUv/o0CSkEpIvQWkXrZfwA0A9Qsm2QmnhFPCKeH0dqbUHX9TzgthcVKYLfIGjtVJuKuAXzkDuuIytVXSXqHGc81E7W0qUa+w6H8DiEqr+Ym1xNobZK3NhyFKQbVBOySuQgTxtJpsKlXu0CkPV/e18kQ40zJmqfZlTUBdeNzNTcJfmiHYvahutCBnOb9errn8DIoxRxlmuO2guoC2wuQL7ni9y/bRM/zsvE3Q22PBY++GoN/gNkLZErl4MBVjxAb4CmZ1FW62AGorm7hUZzkVv1g0uoqooRj5Nadgg/AKJoASQAmgy0kkGLv7i5WeLcKc6MN7MC6u5uDPfwvTSOX0kC/8bDabodQeiQj+t7Iqf+1wbN9NOF4hHbAZhKSUAFGWKHutlP3h0fEJ23/66MX+47/4UBQo2u+Xu0q1umWcirbrpVf6eLm3YCZAbCrYcqiRwq2m1pwG2KYolKJQ4iPxcSOj0HYHCLrOKPQyNO5QDEoxKDGWGHvHUqUCV9uAJFCJcDNVKLAMMz2I/NHlPtYQruJ0lbd9FGgu2BAvr2RTN2PtUvNuMvkqO1lvBpNpYotoTbS+seL+TwNifqtZ2S/4ZiGlCihVQGAkMN6FGf/1bkp6+ZQWZVoJn4RPwucN4RP0JEEqcyxNtTBOj/MUNIE/a/XYCxjlT0HNDfhX6Wkry/S5K1bdO3jYdQSWIOsLwHjRlvpCgXleQMGvQRb0GFSm4e2wKj824xbk46t51YqwcZuQVM/q2G5c9JCnYNxaWM8UNH9R2dkehcZH4vNMAiSEP/En+PEDAQTwxafoVSUyS+54K89tDg7wCX7ggbj4d6j1ZMrNZDGWMPhK1OqcCtZQ30oUIYpcTJEjMcW3mtkLspG4t5ESrkgMWoZJQnwxggaMwRHsKxtUiWaJLzuFYkvBBHPrt/GY8onfVal4sa8VKh2CCkGFoLJxUDmBgBgXVpQ1MGXsEfZVw+neBjsJyzoEzv3BbfkZlyl+dg0lOK9+MSjmNhlqbuL1sqTVJZgQTAgmGzrb+QZO4pYszjMiARcZ4legsnwILU/Q6uZef7nC8ZDF1XZsH+ctgxW7HNXAcKiNQvdpo1Cet2qya0XMzg0T5qQqYv9tLalg9Kdzh54hVbAnuBw/o1lM6KFaF0wp9uBBf3p5cljDHRRriz1xagusg/+D1/DUp1u8QS/nM4hZxKz7x6zDwjPKLbv2MO7Z7vSXIdFUwwhLLJelHhWhkMoxY8OOHn/fb7bbzWaD/SjFrFwEG6Kl1QDp2xpiEbOIWcSszWTWIjl0CbU6zW6707ltarXahC3CFmHrvgwPUzk0HGfBwAgqWetOZQrsSCggknQr3yEuf1eMCyufIK6ejYWlkSHhinBFuLoeXI0xfYUTbHhlmecuPkPSanZ6lW83HQDJYu4ExjQVmvnwSrqQIRZL493Hqn004fXyqkm8Il4Rr+5ffcDQaB4P8ekrhZsGgyv7ccj1HJo7FuEkCNsKh3V8toRWYbgzMbQQt31STLBegm0TwYhgRLD7RzCMve44u7rELmIXseveLJEpzLD0oqPH3+9ub2/3WssyiDzDMaLfOuHB8gu/DxrsuNj/vZr0WrFd+MVZmaovtjBeL7z6BC+CF8Hrnmfm2927kpqnmUTiFfHq3vAKTExL5byL+P4/MRBgeQ0fwDHuF8VE4dsPxUq75bARF9S191pddvi8wfZxmSdquiiO0Kqqd/OgssK0HEaWy2dwQSl+gIDARmAjsBHYrjcDVt057IJM2Mpc4yanwloEMYIYQey+lqe2+pXyVJ/Zv7gytdfe6e70b7syleYdCVgELKqnR3HNuBPm0mL67m5/Z5eWABGyCFmErFtCVnslyLrKGqBuc6fXblGkRdgibBG2bihxDz4C9Hl5dNDa3tntt5e1ETaRWYab3YUPOC8MDLT7XCu/Mf6J4dEklNNb3N3cm3Wis8z7KNjuLMG90TG1Bf2LvIZFeOKa01tUaE8QI4jdG4gty+v/+PKg3hWWB/0rl9Fknxu3/iVCVOdF5CJy3cPwq7ez3dq+UvhVJdbGhmC7xDHiGHHsvma/un80+dXb2d1ptW87+UXjRqIWUes+LxFq7TbbzZ07uUSoR/AieBG8KORa3XDikgnH1na306WYi7BF2CJsbUIt6ue3d+70uv32rY8TW1RAT9AiaN2/NP1up7/b3r1Smr5zF9L0tOPzNwqy/wMb9SY6fvsAAA=="

def unpack_jsonl(b64_str):
    raw_bytes = gzip.decompress(base64.b64decode(b64_str))
    lines = raw_bytes.decode('utf-8').strip().split('\n')
    items = [json.loads(line) for line in lines if line.strip()]
    return items

train_raw = unpack_jsonl(TRAIN_B64)
eval_raw = unpack_jsonl(EVAL_B64)
test_raw = unpack_jsonl(TEST_B64)

# Format each item into ChatML prompt string using model tokenizer
def format_to_chatml(items):
    samples = []
    for item in items:
        formatted = tokenizer.apply_chat_template(item["messages"], tokenize=False)
        samples.append({"text": formatted})
    return Dataset.from_list(samples)

train_dataset = format_to_chatml(train_raw)
eval_dataset = format_to_chatml(eval_raw)

print(f"✅ Successfully ingested 508 synthetic scam samples:")
print(f"   - Training Set:   {len(train_dataset)} instruction pairs")
print(f"   - Validation Set: {len(eval_dataset)} instruction pairs")
print(f"   - Holdout Test:   {len(test_raw)} verification pairs")

## Step 5: Supervised Fine-Tuning with SFTTrainer
We fine-tune the model using cosine learning rate scheduling for 2 epochs on the T4 GPU (~10-15 minutes).

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.05,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        evaluation_strategy = "steps",
        eval_steps = 25,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "ghostsignal_smollm3_outputs",
        report_to = "none"
    ),
)

print("🚀 Starting fine-tuning of SmolLM3-3B on GPU...")
trainer_stats = trainer.train()
print("✅ Training completed successfully!")

## Step 6: Holdout Test Set Evaluation & Live Inference
Evaluate the fine-tuned model against the 50 holdout test samples and test a live obfuscated scam message.

In [ ]:
import time
FastLanguageModel.for_inference(model)

# 1. Test against holdout samples
correct_classifications = 0
total_tested = min(20, len(test_raw)) # Test first 20 holdouts for fast verification

print(f"Evaluating model against {total_tested} holdout test samples...")
start_time = time.time()

for i in range(total_tested):
    test_item = test_raw[i]
    user_msg = test_item["messages"][1]["content"]
    expected_payload = json.loads(test_item["messages"][2]["content"])
    expected_risk = expected_payload["risk_level"]

    prompt = tokenizer.apply_chat_template([
        test_item["messages"][0],
        test_item["messages"][1]
    ], tokenize=False, add_generation_prompt=True)

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=250, use_cache=True)
    pred_str = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    if expected_risk in pred_str:
        correct_classifications += 1

elapsed = time.time() - start_time
accuracy = (correct_classifications / total_tested) * 100
avg_latency_ms = (elapsed / total_tested) * 1000

print(f"✅ Holdout Test Accuracy: {accuracy:.1f}%")
print(f"✅ Average Inference Latency: {avg_latency_ms:.1f} ms/sample")

# 2. Live inference with dotted keyword obfuscation
live_scam = "URGENT: Your Northstar b.a.n.k account is blocked due to missing k.y.c. Verify immediately at https://parcel-fee.example.test/pay"
live_prompt = tokenizer.apply_chat_template([
    {"role": "system", "content": "You are GhostSignal AI, an offline cybersecurity scam detection model. Output strict JSON with risk_level, risk_score, scam_type, red_flags, explanation, recommended_action."},
    {"role": "user", "content": f"Analyze this message for scam indicators:\n\"\"\"{live_scam}\"\"\""}
], tokenize=False, add_generation_prompt=True)

inputs = tokenizer([live_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=300, use_cache=True)
live_verdict = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("\n--- Live Scam Analysis Output ---")
print(live_verdict)

## Step 7: Quantize to GGUF (Q4_K_M ~1.82 GB)
Quantize the fine-tuned model directly into **GGUF Q4_K_M** (4-bit medium). This reduces the model size from 6.0 GB down to ~1.82 GB (3.3x compression) for 100% offline CPU execution.

In [ ]:
output_dir = "ghostsignal_smollm3_3b_q4km"
print(f"Exporting fine-tuned model to GGUF Q4_K_M in '{output_dir}'...")
model.save_pretrained_gguf(output_dir, tokenizer, quantization_method = "q4_k_m")
print("✅ Quantization to GGUF Q4_K_M completed successfully!")

## Step 8: Download Model to Your Computer
Run this cell to download the quantized GGUF weights directly through your browser.

In [ ]:
import glob
import os
from google.colab import files

gguf_files = glob.glob("ghostsignal_smollm3_3b_q4km/*.gguf")
if gguf_files:
    target_file = gguf_files[0]
    size_mb = os.path.getsize(target_file) / (1024 * 1024)
    print(f"📦 Quantized GGUF File: {target_file} ({size_mb:.2f} MB)")
    print("⬇️ Initiating browser download...")
    files.download(target_file)
else:
    print("❌ GGUF file not found. Check the ghostsignal_smollm3_3b_q4km directory.")

## Step 9: 100% Offline Local Deployment in GhostSignal
Once downloaded to your device:
1. Rename the downloaded file to: `smollm3-3b-q4.gguf`
2. Place it into your GhostSignal project folder at:
   ```
   backend/models/smollm3-3b-q4.gguf
   ```
3. Restart the GhostSignal backend:
   ```bash
   python backend/main.py
   ```
4. GhostSignal automatically detects the local quantized weights and runs all contextual scam reasoning offline without internet!